In [25]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [26]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
from pathlib import Path
print("library loaded succesfully")

library loaded succesfully


In [27]:
st.set_page_config(
    page_title="Retail Analytics Dashboard",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Retail Customer Analytics Dashboard")
st.markdown(
    "Sales performance, customer segmentation and churn prediction"
)

2026-11-16 12:34:46.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 12:34:46.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 12:34:46.681 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 12:34:46.683 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 12:34:46.684 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [28]:
BASE_DIR = Path.cwd()

CUSTOMER_FILE = BASE_DIR / "customers.csv"

TRANSACTION_FILE = BASE_DIR / "transactions.csv"

FEATURE_FILE = BASE_DIR / "customer_level_features.csv"

MODEL_FILE = BASE_DIR / "retail_churn_model.pkl"

In [29]:
def find_column(df, candidates, required=True):

    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(
            f"Required column not found. "
            f"Expected one of: {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


def clean_customer_id(series):

    return series.astype("string").str.strip()


def money(value):

    return f"{value:,.2f}"


In [34]:
customer =r"C:\Users\RSC\Desktop\data sheets\project\data science project\customers.csv"
customers =pd.read_csv(customer)
customers.head()

,CustomerID,Age,Gender,Country,City,SignupDate,Churned,ChurnDefinition,SnapshotDate
0,C00001,37,Female,India,Hyderabad,2023-06-20,0,1 = no purchase in label window; 0 = at least ...,2025-01-01
1,C00002,42,Male,United Kingdom,Manchester,2023-09-15,0,1 = no purchase in label window; 0 = at least ...,2025-01-01
2,C00003,44,Male,United Kingdom,Manchester,2023-10-14,1,1 = no purchase in label window; 0 = at least ...,2025-01-01
3,C00004,51,Female,India,Chennai,2023-05-06,1,1 = no purchase in label window; 0 = at least ...,2025-01-01
4,C00005,56,Male,India,Hyderabad,2023-02-21,0,1 = no purchase in label window; 0 = at least ...,2025-01-01


In [38]:
feature = r"C:\Users\RSC\Desktop\data sheets\project\data science project\customer_level_features.csv"
features = pd.read_csv(feature)
features.head()

,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,PurchaseTrend,CustomerTenure,Churn
0,C00001,33,2,708.23,354.115,0.0,98,0
1,C00002,41,3,1705.44,568.480,1.0,130,0
2,C00004,96,1,199.66,199.660,-1.0,96,0
3,C00006,74,2,1152.75,576.375,0.0,221,0
4,C00009,42,1,54.88,54.880,NaN,42,1


In [41]:
transaction=r"C:\Users\RSC\Desktop\data sheets\project\data science project\transaction.csv"
transactions =pd.read_csv(transaction)
transactions.head()

,InvoiceNo,CustomerID,ProductID,Quantity,UnitPrice,InvoiceDate,LineRevenue
0,INV500001,C00001,P1002,1,54.35,06-07-2024,54.35
1,INV500001,C00001,P1012,2,111.43,06-07-2024,222.86
2,INV500002,C00001,P1018,2,10.47,23-07-2024,20.94
3,INV500003,C00001,P1006,3,117.15,10-09-2024,351.45
4,INV500004,C00001,P1022,3,44.48,31-10-2024,133.44


In [56]:
transactions["InvoiceDate"] = pd.to_datetime(
    transactions["InvoiceDate"],
    errors="coerce"
)

In [43]:
models = r"C:\Users\RSC\Desktop\data sheets\project\data science project\customer_churn_predictions.csv"
model =pd.read_csv(models)
model.head()

,Unnamed: 0,AverageOrderValue,Monetary,PurchaseTrend,Actual_Churn,Predicted_Churn,Churn_Probability,Churn_Risk
0,10,512,512,-1,0,0,0.472140,Medium Risk
1,38,160,321,0,0,1,0.632495,Medium Risk
2,18,342,684,0,0,1,0.550726,Medium Risk
3,32,238,238,0,1,1,0.606642,Medium Risk
4,33,822,822,-1,1,0,0.374175,Medium Risk


In [50]:
print(customer_data.columns.tolist())

['CustomerID', 'Recency', 'Frequency', 'Monetary', 'AverageOrderValue', 'PurchaseTrend', 'CustomerTenure', 'Churn', 'Country']


In [57]:
@st.cache_data
def load_data():

    customers = pd.read_csv(CUSTOMER_FILE)
    transactions = pd.read_csv(TRANSACTION_FILE)
    features = pd.read_csv(FEATURE_FILE)

    customer_id_col = find_column(
        customers,
        ["CustomerID", "CustomerId", "customer_id"]
    )

    transaction_customer_col = find_column(
        transactions,
        ["CustomerID", "CustomerId", "customer_id"]
    )

    feature_customer_col = find_column(
        features,
        ["CustomerID", "CustomerId", "customer_id"]
    )

    transaction_date_col = find_column(
        transactions,
        [
            "InvoiceDate",
            "TransactionDate",
            "OrderDate",
            "PurchaseDate",
            "Date"
        ]
    )

    transaction_sales_col = find_column(
        transactions,
        [
            "LineRevenue",
            "Revenue",
            "Sales",
            "Amount",
            "TotalAmount"
        ],
        required=False
    )

    if transaction_sales_col is None:

        quantity_col = find_column(
            transactions, ["Quantity"]
        )

        price_col = find_column(
            transactions, ["UnitPrice", "Price"]
        )

        transactions["Sales"] = (
            pd.to_numeric(
                transactions[quantity_col],
                errors="coerce"
            )
            *
            pd.to_numeric(
                transactions[price_col],
                errors="coerce"
            )
        )

        transaction_sales_col = "Sales"

    customers[customer_id_col] = clean_customer_id(
        customers[customer_id_col]
    )

    transactions[transaction_customer_col] = clean_customer_id(
        transactions[transaction_customer_col]
    )

    features[feature_customer_col] = clean_customer_id(
        features[feature_customer_col]
    )

    transactions[transaction_date_col] = pd.to_datetime(
        transactions[transaction_date_col],
        errors="coerce"
    )

    transactions[transaction_sales_col] = pd.to_numeric(
        transactions[transaction_sales_col],
        errors="coerce"
    )

    features["Churned"] = pd.to_numeric(
        features["Churned"],
        errors="coerce"
    )

    transactions = transactions.dropna(
        subset=[
            transaction_date_col,
            transaction_sales_col,
            transaction_customer_col
        ]
    )

    customers = customers.rename(
        columns={customer_id_col: "CustomerID"}
    )

    transactions = transactions.rename(
        columns={
            transaction_customer_col: "CustomerID",
            transaction_date_col: "TransactionDate",
            transaction_sales_col: "Sales"
        }
    )

    features = features.rename(
        columns={feature_customer_col: "CustomerID"}
    )

    return customers, transactions, features


@st.cache_resource
def load_model():

    if not MODEL_FILE.exists():
        raise FileNotFoundError(
            "retail_churn_model.pkl was not found."
        )

    return joblib.load(MODEL_FILE)

try:

    customers, transactions, features = load_data()
    model = load_model()

except Exception as error:

    st.error(f"Unable to load application data: {error}")
    st.stop()

country_col = find_column(
    customers,
    ["Country", "country", "Region"],
    required=False
)

if country_col:

    customer_info = customers[
        ["CustomerID", country_col]
    ].drop_duplicates("CustomerID")

    customer_info = customer_info.rename(
        columns={country_col: "Country"}
    )

else:

    customer_info = customers[
        ["CustomerID"]
    ].drop_duplicates()


customer_data = features.merge(
    customer_info,
    on="CustomerID",
    how="left"
)

customer_data = customer_data.drop_duplicates(
    subset=["CustomerID"]
)

print("Transaction columns:", transactions.columns.tolist())
print("Customer info columns:", customer_info.columns.tolist())

if "Country" not in transactions.columns:
    transactions = transactions.merge(
        customer_info[["CustomerID", "Country"]].drop_duplicates(
            subset=["CustomerID"]
        ),
        on="CustomerID",
        how="left",
        validate="many_to_one"
    )

print("Merge completed successfully!")

st.sidebar.header("Dashboard Filters")

if hasattr(model, "feature_names_in_"):

    model_features = list(model.feature_names_in_)

else:

    excluded_columns = [
        "CustomerID",
        "Churned",
        "FeatureCutoffDate",
        "ChurnLabelStartDate",
        "ChurnLabelEndDate",
        "SignupDate",
        "FirstPurchaseDate",
        "LastPurchaseDate"
    ]

    model_features = [
        col for col in features.columns
        if col not in excluded_columns
    ]

missing_features = [
    col for col in model_features
    if col not in customer_data.columns
]

if missing_features:

    st.error(
        "The model requires columns missing from the feature dataset: "
        + ", ".join(missing_features)
    )

    st.stop()


x_customer = customer_data[model_features].copy()

try:

    churn_probabilities = model.predict_proba(
        x_customer
    )[:, 1]

    customer_data["Churn"] = churn_probabilities

    customer_data["ChurnPrediction"] = (
        churn_probabilities >= 0.50
    ).astype(int)

except Exception as error:

    st.error(f"Prediction failed: {error}")
    st.stop()


def risk_category(score):

    if score >= 0.70:
        return "High Risk"

    elif score >= 0.40:
        return "Medium Risk"

    else:
        return "Low Risk"


customer_data["RiskCategory"] = (
    customer_data["Churn"].apply(risk_category)
)


if "Monetary" in customer_data.columns:

    q1 = customer_data["Monetary"].quantile(0.33)
    q2 = customer_data["Monetary"].quantile(0.67)

    def segment_customer(value):

        if pd.isna(value):
            return "Unknown"

        elif value >= q2:
            return "High Value"

        elif value >= q1:
            return "Regular Value"

        else:
            return "Low Value"

    customer_data["CustomerSegment"] = (
        customer_data["Monetary"].apply(segment_customer)
    )

else:

    customer_data["CustomerSegment"] = "Unknown"

min_date = transactions["InvoiceDate"].min().date()
max_date = transactions["InvoiceDate"].max().date()

date_range = st.sidebar.date_input(
    "Select Sales Date Range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

countries = sorted(
    transactions["Country"].dropna().unique().tolist()
) if "Country" in transactions.columns else []

selected_countries = st.sidebar.multiselect(
    "Select Country",
    options=countries,
    default=countries
)

churn_options = ["All", "Churned", "Retained"]

selected_churn = st.sidebar.selectbox(
    "Churn Status",
    churn_options
)

risk_options = [
    "All",
    "High Risk",
    "Medium Risk",
    "Low Risk"
]

selected_risk = st.sidebar.selectbox(
    "Churn_Risk",
    risk_options
)

segment_options = (
    customer_data["CustomerSegment"].dropna().unique().tolist()
)

selected_segments = st.sidebar.multiselect(
    "Customer Segment",
    options=segment_options,
    default=segment_options
)

if len(date_range) != 2:

    st.warning("Please select both start and end dates.")
    st.stop()

start_date, end_date = date_range

if start_date > end_date:

    st.error("Start date must be before end date.")
    st.stop()

filtered_transactions = transactions[
    (transactions["InvoiceDate"].dt.date >= start_date)
    &
    (transactions["InvoiceDate"].dt.date <= end_date)
].copy()

if "Country" in filtered_transactions.columns:

    filtered_transactions = filtered_transactions[
        filtered_transactions["Country"].isin(
            selected_countries
        )
    ]

filtered_customers = customer_data.copy()

if selected_churn == "Churned":

    filtered_customers = filtered_customers[
        filtered_customers["Churned"] == 1
    ]

elif selected_churn == "Retained":

    filtered_customers = filtered_customers[
        filtered_customers["Churned"] == 0
    ]

if selected_risk != "All":

    filtered_customers = filtered_customers[
        filtered_customers["RiskCategory"] == selected_risk
    ]

filtered_customers = filtered_customers[
    filtered_customers["CustomerSegment"].isin(
        selected_segments
    )
]

if "Country" in filtered_customers.columns:

    filtered_customers = filtered_customers[
        filtered_customers["Country"].isin(
            selected_countries
        )
    ]

2026-11-16 14:20:05.058 No runtime found, using MemoryCacheStorageManager
2026-11-16 14:20:05.062 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.063 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.068 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.070 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.071 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:20:05.072 Thread 'MainThread':

Transaction columns: ['InvoiceNo', 'CustomerID', 'ProductID', 'Quantity', 'UnitPrice', 'InvoiceDate', 'LineRevenue', 'Country_x', 'Country_y', 'Country']
Customer info columns: ['CustomerID', 'Country']
Merge completed successfully!


In [61]:
st.header("Business Overview")

total_sales = filtered_transactions["LineRevenue"].sum()

total_orders = len(
    filtered_transactions[
        ["CustomerID", "InvoiceDate"]
    ].drop_duplicates()
)

total_customers = filtered_customers["CustomerID"].nunique()

print(filtered_customers.columns.tolist())

average_order_value = (
    total_sales / total_orders
    if total_orders > 0 else 0
)

labeled_customers = filtered_customers.dropna(
    subset=["Churn"]
)

churn_rate = (
    labeled_customers["Churn"].mean() * 100
    if len(labeled_customers) > 0 else 0
)

high_risk_count = (
    filtered_customers["RiskCategory"] == "High Risk"
).sum()

k1, k2, k3, k4, k5 = st.columns(5)

k1.metric(
    "Total Sales",
    money(total_sales)
)

k2.metric(
    "Total Orders",
    f"{total_orders:,}"
)

k3.metric(
    "Customers",
    f"{total_customers:,}"
)

k4.metric(
    "Average Order Value",
    money(average_order_value)
)

k5.metric(
    "Observed Churn Rate",
    f"{churn_rate:.2f}%"
)

st.caption(
    f"High-risk customers in the filtered customer population: "
    f"{high_risk_count:,}"
)

2026-11-16 14:24:40.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.276 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.286 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.288 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:24:40.291 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

['CustomerID', 'Recency', 'Frequency', 'Monetary', 'AverageOrderValue', 'PurchaseTrend', 'CustomerTenure', 'Churn', 'Country', 'RiskCategory', 'CustomerSegment']


DeltaGenerator()

In [64]:
st.header("📈 Sales Trend Analysis")

frequency = st.selectbox(
    "Sales Trend Frequency",
    ["Daily", "Weekly", "Monthly"]
)

if frequency == "Daily":
    sales_trend = (
        filtered_transactions
        .set_index("InvoiceDate")["LineRevenue"]
        .resample("D")
        .sum()
        .reset_index()
    )

elif frequency == "Weekly":
    sales_trend = (
        filtered_transactions
        .set_index("InvoiceDate")["LineRevenue"]
        .resample("W-SUN")
        .sum()
        .reset_index()
    )

else:
    sales_trend = (
        filtered_transactions
        .set_index("InvoiceDate")["LineRevenue"]
        .resample("MS")
        .sum()
        .reset_index()
    )

if not sales_trend.empty:

    fig = px.line(
        sales_trend,
        x="InvoiceDate",
        y="LineRevenue",
        title=f"{frequency} Sales Trend",
        markers=True
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

else:

    st.info("No sales records found for the selected filters.")

2026-11-16 14:26:21.309 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.312 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.313 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.315 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.316 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.318 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:21.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [65]:
st.header("👥 Customer Segmentation")

col1, col2 = st.columns(2)

with col1:

    segment_counts = (
        filtered_customers["CustomerSegment"]
        .value_counts()
        .reset_index()
    )

    segment_counts.columns = [
        "Segment",
        "Customers"
    ]

    fig_segment = px.pie(
        segment_counts,
        names="Segment",
        values="Customers",
        title="Customer Distribution by Segment",
        hole=0.4
    )

    st.plotly_chart(
        fig_segment,
        use_container_width=True
    )

with col2:

    if "Monetary" in filtered_customers.columns:

        segment_revenue = (
            filtered_customers
            .groupby("CustomerSegment")["Monetary"]
            .sum()
            .reset_index()
        )

        fig_revenue = px.bar(
            segment_revenue,
            x="CustomerSegment",
            y="Monetary",
            title="Historical Customer Spend by Segment",
            text_auto=".2s"
        )

        st.plotly_chart(
            fig_revenue,
            use_container_width=True
        )

2026-11-16 14:26:28.300 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.301 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.302 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.303 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.303 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.388 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.390 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:26:28.391 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [68]:
st.header("⚠️ Customer Churn Risk Analysis")

col1, col2 = st.columns(2)

with col1:

    risk_counts = (
        filtered_customers["RiskCategory"]
        .value_counts()
        .reset_index()
    )

    risk_counts.columns = [
        "Churn",
        "Customers"
    ]

    fig_risk = px.bar(
        risk_counts,
        x="Churn",
        y="Customers",
        color="Churn",
        title="Customers by Predicted Churn Risk"
    )

    st.plotly_chart(
        fig_risk,
        use_container_width=True
    )

with col2:

    fig_probability = px.histogram(
        filtered_customers,
        x="RiskCategory",
        nbins=20,
        title="Churn Probability Distribution",
        labels={
            "RiskCategory": "Predicted Churn Probability"
        }
    )

    st.plotly_chart(
        fig_probability,
        use_container_width=True
    )

2026-11-16 14:27:53.018 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.020 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.022 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.023 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.024 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.065 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:53.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [69]:

st.header("🎯 Recommended Customer Retention Actions")


def recommend_action(row):

    risk = row["RiskCategory"]
    segment = row["CustomerSegment"]

    if risk == "High Risk" and segment == "High Value":

        return (
            "Priority personal outreach, investigate service issues "
            "and offer a tailored retention benefit."
        )

    elif risk == "High Risk":

        return (
            "Send a targeted re-engagement campaign and "
            "consider a limited-time incentive."
        )

    elif risk == "Medium Risk" and segment == "High Value":

        return (
            "Provide loyalty rewards, personalized product "
            "recommendations and proactive follow-up."
        )

    elif risk == "Medium Risk":

        return (
            "Send personalized recommendations and "
            "monitor purchase activity."
        )

    elif risk == "Low Risk" and segment == "High Value":

        return (
            "Maintain loyalty benefits and provide "
            "exclusive product updates."
        )

    else:

        return (
            "Use low-cost engagement campaigns and "
            "monitor future purchasing behaviour."
        )

2026-11-16 14:27:58.435 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:27:58.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [80]:
filtered_customers.head()

,CustomerID,Recency,Frequency,Monetary,AverageOrderValue,PurchaseTrend,CustomerTenure,Churn,Country,RiskCategory,CustomerSegment,RecommendedAction
0,C00001,33,2,708.23,354.115,0.0,98,0,India,Low Risk,Regular Value,Use low-cost engagement campaigns and monitor ...
1,C00002,41,3,1705.44,568.480,1.0,130,0,United Kingdom,Low Risk,High Value,Maintain loyalty benefits and provide exclusiv...
2,C00004,96,1,199.66,199.660,-1.0,96,0,India,Low Risk,Low Value,Use low-cost engagement campaigns and monitor ...
3,C00006,74,2,1152.75,576.375,0.0,221,0,United Kingdom,Low Risk,High Value,Maintain loyalty benefits and provide exclusiv...
4,C00009,42,1,54.88,54.880,NaN,42,1,India,High Risk,Low Value,Send a targeted re-engagement campaign and con...


In [82]:

filtered_customers = filtered_customers.copy()

filtered_customers["RecommendedAction"] = (
    filtered_customers.apply(
        recommend_action,
        axis=1
    )
)

display_columns = [
    "CustomerID",
    "CustomerSegment",
    "Churn",
    "RiskCategory",
    "RecommendedAction"
]

if "Monetary" in filtered_customers.columns:
    display_columns.insert(2, "Monetary")

if "RecencyDays" in filtered_customers.columns:
    display_columns.insert(3, "RecencyDays")

In [83]:
st.dataframe(
    filtered_customers[display_columns]
    .sort_values(
        "Churn",
        ascending=False
    ),
    use_container_width=True,
    hide_index=True
)

st.subheader("Why are these actions recommended?")

st.markdown("""
- **High risk + high value:** These customers have high predicted churn
  probabilities and high historical spend. Personal retention efforts
  may be worth prioritizing.
- **High risk + regular/low value:** Use scalable, lower-cost campaigns
  rather than expensive individual interventions.
- **Medium risk:** Encourage repeat purchases and monitor engagement.
- **Low risk:** Focus on loyalty and customer satisfaction.

These are business recommendations based on risk and historical
spending, not proof that a particular action will prevent churn.
Test campaign effectiveness before scaling.
""")

2026-11-16 14:33:30.505 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:33:30.506 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:33:30.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:33:30.508 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:33:30.509 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:33:30.510 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [85]:
st.header("📥 Export Customer Predictions")
download_columns = [
    "CustomerID",
    "Churn",
    "RiskCategory",
    "CustomerSegment",
    "RecommendedAction"
]

if "Churned" in filtered_customers.columns:
    download_columns.append("Churned")

if "Monetary" in filtered_customers.columns:
    download_columns.append("Monetary")

download_data = filtered_customers[
    download_columns
].copy()

download_data["Churn"] = (
    download_data["Churn"] * 100
).round(2)

csv_data = download_data.to_csv(
    index=False
).encode("utf-8")

st.download_button(
    label="Download Customer Churn Predictions",
    data=csv_data,
    file_name="customer_churn_predictions.csv",
    mime="text/csv"
)

2026-11-16 14:34:05.766 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.767 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.783 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.785 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.786 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:05.788 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

False

In [87]:

st.header("ℹ️ Data Limitations and Model Warnings")

st.warning("""
**Important limitations**

1. The project dataset is synthetic and may not reflect real retail
   customer purchasing behaviour.

2. The churn label is an inactivity proxy: customers with no purchase
   during the defined future label window are marked as churned.
   It does not mean the customer formally cancelled an account.
3. Churn probabilities are model estimates, not guaranteed outcomes.

4. High, medium and low risk categories use illustrative thresholds
   of 70% and 40%. They should be validated against business costs
   and model calibration.

5. Customer segments are based on historical monetary spend and
   dataset-specific quantiles, not permanent customer characteristics.

6. Sales are transaction-line revenue aggregated by transaction date.
   They may not represent net revenue after returns, discounts,
   taxes or operating costs.

7. Historical churn labels and current model predictions are different:
   the observed churn rate is not the same as the percentage predicted
   to churn.

8. The sales date filter and customer feature cutoff are separate.
   Customer risk scores are based on the saved feature snapshot,
   not recalculated from the selected sales period.

9. This dashboard does not establish whether a retention campaign
   causes a reduction in churn.
""")

st.info(
    "Use real, permissioned customer data and validate the model "
    "before making operational decisions. Restrict access to "
    "customer-level predictions."
)

st.caption(
    "Retail Customer Analytics | Sales and Churn Prediction"
)


2026-11-16 14:34:17.627 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.629 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.631 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.632 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.633 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-16 14:34:17.634 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()